In [8]:
#TODO make sure this renders in the github repo

In [9]:
# - [ ] #TODO add info about warmup steps etc for each model.

# Configurations For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)

- **Layers:** Noted as **num_layers**. How many Decoder layers the model has.
- **Model Dimensions:** Also noted as **d_model** or $d_{model}$. It represents the expected features for each token in the input and output sequences.
  - Example: when the Token Embeddings layer happen, each token is converted into a dense vector of size $d_{model}$.
- **FFN Dimensions:** Noted as **d_ff**. Is the hidden size of the Feed Forward SwiGLU sub-layer. We use this to expand the linear layers for SwiGLU, which gives the model more parameters to learn non-linear functions. 
- **Attention Heads:** This the the total number of Query heads, note K & V use the Key/Value Heads hyperparameter.
  - If d_model = $4{,}096$ and attn_heads = $32$, each individual head processes a vector of size $4{,}096/32 = 128$ 
- **Key/Value Heads:** Noted as **num_kv_heads**. The number specific to the Key and Value, in older models like Llama 1, this didn't exist because the number of K and V heads were the same as the number of Q heads.
- [ ] # TODO: Peak Learning Rate

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length:** The model's short-term memory. Represents how much words/tokens the model can remember at one time. This changes over time, if you give the model content that is larger than its context window length, it forget the beginning of the content to make room for the end of the content. So, as you change with the model, it loses track of earlier context that no longer fits its context window.

- **Positional Embeddings** The frequency that the Q and K tokens are rotated by, to give them positional information.

The [Llama-3.1 8B tokenizer tokens](https://huggingface.co/meta-llama/Llama-3.1-8B/blob/main/tokenizer_config.json) consists of $128{,}255$ tokens.
- **Standard tokens:** IDs $0$ through $127{,}999$ ($128{,}000$ total) are the standard tokens.
- **Special tokens:** IDs $128{,}000$ through $128{,}255$ ($256$ total) are the special tokens.
- **From Llama 3 paper:** "We use a vocabulary with 128K tokens. Our token vocabulary combines 100K tokens from the tiktoken tokenizer with 28K additional tokens to better support non-English languages. Compared to the Llama 2 tokenizer, our new tokenizer improves compression rates on a sample of English data from 3.17 to 3.94 characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding 28K tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

In [10]:
import os
import EasyJupyter
from pathlib import Path
import torch

class BaseConfig:
    """
    #TODO add docstring for what each attribute is for.

    The is the base config class that all models inherit from.
    """


    # All Llama models use the same below parameters, except for my scale down model.
    num_kv_heads = 8
    activation_function = "SwiGLU"
    vocab_size = 128_000
    pos_embeddings_freq = 500_000 # RoPE
    context_len = None #TODO make each model have a differnt context length, also make sure that all the meta models use the same amount

    # Force child classes to implement the below attributes
    _required_attributes: dict[str, type] = {
        "model_name": str,
        "num_layers": int, # How many decoder layers the model has.
        "d_model": int,
        "d_ff": int, # How much to expand the SwiGLU linear to.
        "attn_heads": int, # The number of independent "Query" viewpoints the attention mechanism splits the data into.
        "norm_eps" : float, # RMSNorm epsilon #TODO set to 1e-5 for most models, maybe higher for my scale down model.
        "peak_lr": float, # The peak learning rate.
        # "batch_size": int, #TODO add batch size
    }

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for attr, expected_type in cls._required_attributes.items():
            value = getattr(cls, attr, None)
            if value is None:
                raise ValueError(f"Missing attribute {attr} in {cls.__name__}")
            if not isinstance(value, expected_type):
                raise TypeError(
                    f"Attribute {attr} in {cls.__name__} must be of type {expected_type}, but got {type(value)}"
                )
        pass



    # ================== Training ==================
    total_training_steps = 1_200_000 # TODO: scale down for my scaled down model
    continue_from_chpt:bool = False # Continue training the model from a check point
    # Checkpoint filename to load! 🚨 NOTE: The same config that was used to train the checkpoint must be used to load it! And you must use the same tokenizer!
    checkpoint_name:str = "" 
    warmup_steps = 8000 # TODO: scale down for my scaled down model



    # ================== Dataset ==================
    # TODO remove any of the below commented out attributes if not needed.
    # max_batch_seq_tokens = 8_000 # Paper: 25_000. Applies sequence limit to an entire batch of sequences.
    # dataset_name = "" #TODO

    special_tokens = {
        "pad_token": {
            "token": "<|pad_token|>",
            "ID": 0,
        },
        "doc_begin_token": {  # The token for the beginning of the text.
            "token": "<|begin_of_doc|>",
            "ID": 1, # TODO is this necessary? Check if I used it
        },
        "doc_end_token": {  # The token for the end of document, it is injected between documents.
            "token": "<|end_of_doc|>",
            "ID": 2,
        },
        "unk_token": {  # The unknown token.
            "token": "<|unk|>",
            "ID": 3,
        },
    }



    # ================== System ==================
    num_workers = 0 # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    def __init__(self):
        # ================== Folder Structure ==================
        self.PROJECT_ROOT = self._find_root()
        print("\nProject Root:", self.PROJECT_ROOT)
        self.DATA_DIR = self.PROJECT_ROOT / "data"
        self.MODEL_DIR = self.PROJECT_ROOT / "model"
        self.folders_to_make = [
            self.DATA_DIR,
            self.MODEL_DIR / "saved_models",
            self.MODEL_DIR / "saved_models" / "tokenizer",
            self.MODEL_DIR / "saved_models" / "pre-trained", # Store Meta-models here
            self.MODEL_DIR / "checkpoints",
        ]

    def _find_root(self) -> Path:
        """Find the root directory of the project, by climbing up the directory tree"""
        curr = Path.cwd().resolve()

        for parent in [curr] + list(curr.parents):
            # if (parent / ".gitignore").exists():
            #     return parent
            if parent.name == "How_to_build_an_LLM":
                return parent
            
        return curr

    def print_config(self):
        # TODO Make the print method in the parent and have the child call it to prints its own config.
        pass
    
    def _setup_folders(self):
        for folder in self.folders_to_make:
            folder.mkdir(parents=True, exist_ok=True)

In [11]:
class Llama3_8B(BaseConfig):
    """
    Llama 3.1 8B Configuration. This model is not multi-modal, it only takes in text!
    """

    model_name = "Llama 3.1 8B"
    num_layers = 32
    d_model = 4_096
    d_ff = 14_336
    attn_heads = 32
    peak_lr = 3e-4
    norm_eps= 1e-5 #TODO set to paper value

    def __init__(self):
        super().__init__()

In [12]:
class Llama3_scaled_down(BaseConfig):
    """
    Scaled down version of the Llama 3 architecture that is trainable on my Mac.
    """
    # TODO play around with the hyperparameters to see what my mac can handle.
    model_name = "Scaled down Llama 3"
    num_layers = 6
    d_model = 256
    d_ff = 1024
    attn_heads = 32
    peak_lr = 3e-4
    context_len = 256
    batch_size = 16
    norm_eps= 1e-5 #TODO set to 

    # TODO for this model the basemodel configs need to be adjusted like vocab_size it to be somewhere between 16,000 to 32,000

    def __init__(self):
        super().__init__()
        print(os.getcwd())

In [13]:
# @i-c
l = Llama3_8B()
l._setup_folders()


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


In [14]:
# TODO add the config for the larger models